# 🛒 Visual Product Search — Part 1
PART 1 — BLIP-2 Captioning + CLIP Fine-tuning
Output: `captions_cache.json`, per-seed checkpoints `clip_seed<N>.pt` → save as dataset for ablation notebooks.
Estimated time: ~6-8 hours on T4 (captioning ~3-4 h batched, fine-tuning ~2-3 h × 3 seeds).

In [1]:
# ── Install ──
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics', 'open_clip_torch', 'transformers', 'accelerate'])

# ── Suppress warnings ──
import warnings, logging, os
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import math, torch, random, json, numpy as np, pandas as pd
from tqdm import tqdm
from PIL import Image
from collections import defaultdict
import torch.nn.functional as F

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00


In [2]:
# ── Config ──
BASE_INPUT = '/kaggle/input/datasets/sartma/deepfashion-inshop'
EVAL_FILE  = f'{BASE_INPUT}/Eval/list_eval_partition.txt'
WORK = '/kaggle/working'
os.makedirs(WORK, exist_ok=True)

IMG_ROOT = None
for c in [f'{BASE_INPUT}/img/img', f'{BASE_INPUT}/img']:
    if os.path.isdir(f'{c}/MEN') or os.path.isdir(f'{c}/WOMEN'):
        IMG_ROOT = c; break
print(f'IMG_ROOT: {IMG_ROOT}')

# ── Replace with your team's roll numbers ──
SEEDS = [14, 37, 106]

FT_EPOCHS     = 15     # increase to 20 if T4 time allows
FT_BATCH_SIZE = 64
FT_LR         = 2e-6
CAPTION_BATCH = 8      # drop to 4 if OOM during captioning
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CAPTION_CACHE = f'{WORK}/captions_cache.json'

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s); torch.backends.cudnn.deterministic = True

IMG_ROOT: /kaggle/input/datasets/sartma/deepfashion-inshop/img/img


In [3]:
# ── Parse splits ──
with open(EVAL_FILE) as f: lines = f.readlines()
data = [l.strip().split() for l in lines[2:] if l.strip()]
df = pd.DataFrame(data, columns=['image_name','item_id','split'])
df['full_path'] = df['image_name'].apply(lambda x: os.path.join(IMG_ROOT, x.replace('img/','',1)))
train_df = df[df.split=='train'].reset_index(drop=True)
query_df = df[df.split=='query'].reset_index(drop=True)
gallery_df = df[df.split=='gallery'].reset_index(drop=True)
print(f'Train:{len(train_df)} Query:{len(query_df)} Gallery:{len(gallery_df)}')
assert os.path.exists(gallery_df.iloc[0]['full_path']), 'Paths broken!'


Train:25882 Query:14218 Gallery:12612


In [4]:
# ── YOLO ──
from ultralytics import YOLO
# ⚠️ If you have a fine-tuned YOLO, replace with:
# yolo_model = YOLO('/kaggle/input/your-dataset/best.pt')
yolo_model = YOLO('/kaggle/input/models/sartma/yolo-finetune-inshop/pytorch/default/1/best.pt')

def yolo_crop(path, model, pad=10):
    img = Image.open(path).convert('RGB')
    try:
        res = model(path, verbose=False)
        boxes = res[0].boxes
        if len(boxes) == 0: return img
        best = max(boxes, key=lambda b: (b.xyxy[0][2]-b.xyxy[0][0])*(b.xyxy[0][3]-b.xyxy[0][1]))
        x1,y1,x2,y2 = map(int, best.xyxy[0]); W,H = img.size
        x1, y1 = max(0, x1-pad), max(0, y1-pad)
        x2, y2 = min(W, x2+pad), min(H, y2+pad)
        if x2 <= x1 or y2 <= y1: return img   # guard against degenerate boxes
        return img.crop((x1, y1, x2, y2))
    except: return img

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
# ── CLIP helpers (model loaded per-seed inside finetune()) ──
import open_clip

def get_image_emb(pil, model, preprocess):
    x = preprocess(pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        e = model.encode_image(x); e = e / e.norm(dim=-1, keepdim=True)
    return e.cpu().float().numpy()[0]

def get_text_emb(text, model, tokenizer):
    t = tokenizer([text]).to(DEVICE)
    with torch.no_grad():
        e = model.encode_text(t); e = e / e.norm(dim=-1, keepdim=True)
    return e.cpu().float().numpy()[0]

def fuse(img_e, txt_e, alpha):
    v = alpha*img_e + (1-alpha)*txt_e
    return v / (np.linalg.norm(v) + 1e-8)

In [6]:
# ── BLIP-2 Captioning ──
from transformers import Blip2Processor, Blip2ForConditionalGeneration

BLIP_ID = 'Salesforce/blip2-opt-2.7b'
print('Loading BLIP-2...')
blip_proc = Blip2Processor.from_pretrained(BLIP_ID, use_fast=False)
blip_mdl  = Blip2ForConditionalGeneration.from_pretrained(
    BLIP_ID,
    torch_dtype=torch.float16,
    device_map={'': 0}   # force entire model onto GPU 0, no cross-GPU split
).eval()
CPROMPT = 'Question: Describe this clothing item including color, type, and style. Answer:'

def gen_captions_batch(pil_list):
    prompts = [CPROMPT] * len(pil_list)
    inp = blip_proc(images=pil_list, text=prompts, return_tensors='pt',
                    padding=True).to(DEVICE, torch.float16)
    with torch.no_grad():
        out = blip_mdl.generate(**inp, max_new_tokens=60, num_beams=3)
    results = []
    for o in out:
        t = blip_proc.decode(o, skip_special_tokens=True)
        results.append(t.split('Answer:')[-1].strip() if 'Answer:' in t else t.strip())
    return results

Loading BLIP-2...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

In [7]:
# ── Generate captions ──
if os.path.exists(CAPTION_CACHE):
    with open(CAPTION_CACHE) as f: captions = json.load(f)
    print(f'Loaded {len(captions)} captions from cache')
else:
    captions = {}
    for label, sub in [('gallery', gallery_df), ('query', query_df)]:
        print(f'Captioning {label} ({len(sub)})...')
        rows = [r for _, r in sub.iterrows() if r['image_name'] not in captions]
        for i in tqdm(range(0, len(rows), CAPTION_BATCH)):
            batch = rows[i:i+CAPTION_BATCH]
            try:
                crops = [yolo_crop(r['full_path'], yolo_model) for r in batch]
                caps  = gen_captions_batch(crops)
                for r, c in zip(batch, caps):
                    captions[r['image_name']] = c
            except Exception as e:
                # fallback: process one-by-one on batch failure (e.g. OOM)
                for r in batch:
                    try:
                        captions[r['image_name']] = gen_captions_batch(
                            [yolo_crop(r['full_path'], yolo_model)])[0]
                    except:
                        captions[r['image_name']] = 'clothing item'
            if len(captions) % 1000 == 0:
                with open(CAPTION_CACHE, 'w') as f: json.dump(captions, f)
    with open(CAPTION_CACHE, 'w') as f: json.dump(captions, f)
    print(f'Saved {len(captions)} captions')

# ── Free BLIP-2 GPU memory before fine-tuning ──
import gc
del blip_mdl, blip_proc
gc.collect(); torch.cuda.empty_cache()
print('BLIP-2 unloaded. GPU memory freed for fine-tuning.')

Captioning gallery (12612)...



100%|██████████| 1577/1577 [1:10:42<00:00,  2.69s/it]


Captioning query (14218)...



100%|██████████| 1778/1778 [1:16:16<00:00,  2.57s/it]


Saved 26830 captions
BLIP-2 unloaded. GPU memory freed for fine-tuning.


## ══════════ FINE-TUNE CLIP ══════════


In [8]:
# ══════════ FINE-TUNE CLIP ══════════
from torch.utils.data import Dataset, DataLoader, Sampler

class PairBatchSampler(Sampler):
    def __init__(self, ds, bs):
        self.bs = bs; self.pn = bs//2; self.id2idx = defaultdict(list)
        for i in range(len(ds)): self.id2idx[ds.df.iloc[i]['item_id']].append(i)
        self.valid = [k for k,v in self.id2idx.items() if len(v)>=2]
        print(f'  PairSampler: {len(self.valid)} valid ids, bs={bs}')
    def __iter__(self):
        ids = list(self.valid); random.shuffle(ids); batch = []
        for iid in ids:
            batch.extend(random.sample(self.id2idx[iid],2))
            if len(batch)>=self.bs: yield batch[:self.bs]; batch=batch[self.bs:]
    def __len__(self): return len(self.valid)//self.pn

class FashionDS(Dataset):
    def __init__(self, df, tr):
        self.df=df; self.tr=tr
        self.id2int={v:i for i,v in enumerate(df['item_id'].unique())}
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row=self.df.iloc[i]
        try: img=yolo_crop(row['full_path'],yolo_model)
        except: img=Image.open(row['full_path']).convert('RGB')
        return self.tr(img), self.id2int[row['item_id']]

def info_nce(embs, labels, temp=0.1):
    embs = F.normalize(embs.float(), dim=1, eps=1e-8)
    sim = torch.matmul(embs, embs.T) / temp
    lr = labels.unsqueeze(0)
    pm = (lr == lr.T).float()
    pm.fill_diagonal_(0)
    sm = torch.eye(sim.size(0), device=sim.device).bool()
    sim = sim.masked_fill(sm, -1e4)       # -1e4 avoids NaN in fp16 paths
    lp = F.log_softmax(sim, dim=1)
    valid_mask = (pm.sum(1) > 0).float()  # skip rows with no positives
    loss_per_row = -(lp * pm).sum(1) / pm.sum(1).clamp(min=1)
    return (loss_per_row * valid_mask).sum() / valid_mask.sum().clamp(min=1)

def finetune(seed):
    ckpt = f'{WORK}/clip_seed{seed}.pt'
    if os.path.exists(ckpt): print(f'  Checkpoint exists seed {seed}'); return ckpt
    print(f'  Fine-tuning seed {seed}...'); set_seed(seed)

    fm, train_tf, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
    fm = fm.float().to(DEVICE)
    for p in fm.parameters(): p.requires_grad = False
    for i, blk in enumerate(fm.visual.transformer.resblocks):
        if i >= 8:
            for p in blk.parameters(): p.requires_grad = True

    ds = FashionDS(train_df, train_tf)
    samp = PairBatchSampler(ds, FT_BATCH_SIZE)
    loader = DataLoader(ds, batch_sampler=samp, num_workers=2, pin_memory=True)

    total_steps = FT_EPOCHS * len(loader)
    warmup_steps = len(loader)   # 1 epoch warmup

    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, fm.parameters()), lr=FT_LR)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / max(1, total_steps - warmup_steps)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    global_step = 0

    best = float('inf')
    for ep in range(FT_EPOCHS):
        fm.train(); tot = 0; n_batches = 0
        for imgs, lbs in tqdm(loader, desc=f'  Ep {ep+1}/{FT_EPOCHS}'):
            imgs, lbs = imgs.to(DEVICE), lbs.to(DEVICE)
            with torch.autocast(device_type=DEVICE, enabled=(DEVICE == 'cuda')):
                embs = fm.encode_image(imgs)
                loss = info_nce(embs, lbs)
            if torch.isnan(loss) or torch.isinf(loss):
                print('  [Warning] NaN/Inf loss, skipping batch'); continue
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fm.parameters(), 1.0)
            opt.step(); sched.step(); global_step += 1
            tot += loss.item(); n_batches += 1
        if n_batches == 0:
            print('  [Warning] Epoch had 0 valid batches!'); continue
        avg = tot / n_batches
        print(f'  Ep {ep+1} loss: {avg:.4f}  lr: {sched.get_last_lr()[0]:.2e}')
        if avg < best: best = avg; torch.save(fm.state_dict(), ckpt)
    print(f'  Best: {best:.4f} saved to {ckpt}')
    return ckpt

print('\n'+'='*60+'\nFINE-TUNING CLIP FOR ALL SEEDS\n'+'='*60)
for seed in SEEDS:
    print(f'\nSeed {seed}')
    finetune(seed)

print('\n✅✅✅ PART 1 COMPLETE ✅✅✅')
print('Outputs saved to WORK:')
print('  captions_cache.json  — BLIP-2 captions for gallery + query')
for seed in SEEDS: print(f'  clip_seed{seed}.pt      — fine-tuned CLIP checkpoint')
print('\nNext: add /kaggle/working as a dataset, then open your ablation notebooks.')
print(f'\nAll files in {WORK}:')
for f in sorted(os.listdir(WORK)): print(f'  {f}')



FINE-TUNING CLIP FOR ALL SEEDS

Seed 14
  Fine-tuning seed 14...


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

  PairSampler: 3985 valid ids, bs=64



  Ep 1/15: 100%|██████████| 124/124 [01:27<00:00,  1.42it/s]


  Ep 1 loss: 2.2238  lr: 2.00e-06


...

  Ep 13 loss: 0.3411  lr: 9.90e-08



  Ep 14/15: 100%|██████████| 124/124 [00:29<00:00,  4.14it/s]


  Ep 14 loss: 0.3231  lr: 2.51e-08



  Ep 15/15: 100%|██████████| 124/124 [00:31<00:00,  3.90it/s]

  Ep 15 loss: 0.3244  lr: 0.00e+00
  Best: 0.3231 saved to /kaggle/working/clip_seed106.pt

✅✅✅ PART 1 COMPLETE ✅✅✅
Outputs saved to WORK:
  captions_cache.json  — BLIP-2 captions for gallery + query
  clip_seed14.pt      — fine-tuned CLIP checkpoint
  clip_seed37.pt      — fine-tuned CLIP checkpoint
  clip_seed106.pt      — fine-tuned CLIP checkpoint

Next: add /kaggle/working as a dataset, then open your ablation notebooks.

All files in /kaggle/working:
  __notebook__.ipynb
  captions_cache.json
  clip_seed106.pt
  clip_seed14.pt
  clip_seed37.pt
